# 🛡️ AI-Powered Hybrid File Encryption & Threat Detection

Welcome to the interactive exploration notebook for the **AI Hybrid File Encryption System**.

### Core Concepts Explored in this Notebook:
1. **Hybrid Cryptography (RSA-2048 + AES-256-GCM + SHA-256):**
   - Asymmetric RSA-OAEP for key exchange and public/private key protection.
   - Symmetric AES-GCM for high-speed file chunk encryption with authenticated tag verification.
   - SHA-256 cryptographic digest for tamper-evident data integrity.
2. **AI Threat Detection (Unsupervised Anomaly Detection with Isolation Forest):**
   - Real-time behavioral scoring of user operations.
   - Anomaly score computation based on decision tree path lengths.
   - Detection of brute-force logins, ransomware decryptions, and abnormal API call bursts.

## 📦 Step 0: Setup & Imports
Import standard libraries, cryptographic primitives, and machine learning components from the repository.

In [ ]:
import os
import sys
import base64
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to path: {project_root}")

--- 
## 🔐 Part 1: Hybrid Cryptography in Action

### Why Hybrid Encryption?
* **Asymmetric encryption (e.g. RSA)** is computationally expensive and has strict payload size limits (e.g., RSA-2048 can encrypt at most 190 bytes with OAEP-SHA256).
* **Symmetric encryption (e.g. AES-256-GCM)** is blazing fast and handles gigabytes of data, but requires both parties to securely share a secret key.
* **Hybrid Cryptography** combines both: A fresh, random 256-bit AES key encrypts the file; the AES key is then encrypted with the recipient's RSA public key.

In [ ]:
from backend.crypto.aes import encrypt_aes_gcm, decrypt_aes_gcm
from backend.crypto.rsa import encrypt_aes_key_rsa, decrypt_aes_key_rsa
from backend.crypto.hashing import calculate_sha256
from backend.crypto.hybrid import encrypt_hybrid, decrypt_hybrid

# Sample sensitive plaintext data
original_data = (
    b"TOP SECRET REPORT: AI-Driven Zero-Trust Encryption Pipeline. "
    b"All transmissions must be authenticated and integrity checked!"
)

print("Original File Payload:")
print(original_data.decode('utf-8'))
print(f"Original SHA-256: {calculate_sha256(original_data)}")

### 1.1 Perform Hybrid Encryption
Notice how `encrypt_hybrid`:
1. Computes the SHA-256 checksum.
2. Encrypts the payload with AES-256-GCM (producing ciphertext, IV, and authentication tag).
3. Wraps the AES key with RSA-2048 OAEP and encodes it to Base64.

In [ ]:
start_enc = time.perf_counter()
hybrid_package = encrypt_hybrid(original_data)
enc_time = (time.perf_counter() - start_enc) * 1000

print(f"✅ Encryption completed in {enc_time:.3f} ms")
print("-" * 60)
print(f"Encrypted AES Key (Base64 RSA-Wrapped):\n{hybrid_package['encrypted_aes_key_b64'][:80]}...")
print(f"Encrypted Payload Preview (Hex):\n{hybrid_package['encrypted_file_payload'][:32].hex()}...")
print(f"Original Checksum (SHA-256):\n{hybrid_package['sha256_hash']}")

### 1.2 Perform Hybrid Decryption & Integrity Verification
Now we reverse the process using `decrypt_hybrid`:
1. Decrypt the wrapped AES key using RSA private key.
2. Decrypt the ciphertext with AES-GCM.
3. Verify that the SHA-256 of the decrypted payload matches the original checksum.

In [ ]:
start_dec = time.perf_counter()
decryption_result = decrypt_hybrid(
    encrypted_payload=hybrid_package["encrypted_file_payload"],
    encrypted_aes_key_b64=hybrid_package["encrypted_aes_key_b64"],
    expected_sha256=hybrid_package["sha256_hash"]
)
dec_time = (time.perf_counter() - start_dec) * 1000

print(f"✅ Decryption completed in {dec_time:.3f} ms")
print(f"Integrity Verified: {decryption_result['integrity_verified']}")
print(f"Decrypted Text: {decryption_result['decrypted_bytes'].decode('utf-8')}")
assert decryption_result['decrypted_bytes'] == original_data
print("\nSUCCESS: Decrypted payload perfectly matches original!")

### 1.3 Tamper Detection Experiment
What happens if an adversary modifies even 1 single byte of the encrypted payload in transit?

In [ ]:
# Tamper with the encrypted bytes
tampered_payload = bytearray(hybrid_package["encrypted_file_payload"])
tampered_payload[15] ^= 0xFF  # Flip bits in byte 15
tampered_payload = bytes(tampered_payload)

tamper_result = decrypt_hybrid(
    encrypted_payload=tampered_payload,
    encrypted_aes_key_b64=hybrid_package["encrypted_aes_key_b64"],
    expected_sha256=hybrid_package["sha256_hash"]
)

print(f"Success: {tamper_result['success']}")
print(f"Integrity Verified: {tamper_result['integrity_verified']}")
print(f"Status Message: {tamper_result['message']}")

---
## 🤖 Part 2: AI Anomaly & Threat Detection (Isolation Forest)

The system uses an **Isolation Forest** machine learning algorithm to monitor user telemetry and identify potential security threats:

### Feature Vector:
| Feature Index | Name | Description |
|---|---|---|
| 0 | `failed_logins` | Number of failed password attempts |
| 1 | `encryption_requests` | Legitimate file encryption count |
| 2 | `decryption_requests` | Legitimate file decryption count |
| 3 | `failed_decryptions` | Failed decryption attempts (e.g. wrong key / corrupted file) |
| 4 | `access_frequency` | Operations performed per minute |

Isolation Forest isolates anomalies instead of profiling normal points. Because anomalies are few and have distinct attribute values, they are isolated closer to the root of the trees.

In [ ]:
from backend.ml.detector import ThreatDetector, get_threat_detector

detector = get_threat_detector()
print(f"ThreatDetector initialized. Is trained: {detector.is_trained}")
print(f"Model parameters:\n{detector.model}")

### 2.1 Inspecting Baseline Training Data Distribution
Let's recreate and inspect the baseline user behavior matrix used for training the anomaly detector.

In [ ]:
np.random.seed(42)

# 450 normal user activity samples
n_samples = 450
normal_failed_logins = np.random.poisson(lam=0.3, size=n_samples)
normal_enc_reqs = np.random.randint(1, 15, size=n_samples)
normal_dec_reqs = np.random.randint(0, 10, size=n_samples)
normal_failed_dec = np.random.poisson(lam=0.1, size=n_samples)
normal_freq = np.random.uniform(0.5, 5.0, size=n_samples)

df_normal = pd.DataFrame({
    'failed_logins': normal_failed_logins,
    'encryption_requests': normal_enc_reqs,
    'decryption_requests': normal_dec_reqs,
    'failed_decryptions': normal_failed_dec,
    'access_frequency': normal_freq
})

df_normal.describe().round(2)

--- 
## 🚨 Part 3: Live Threat Simulations & Classification

Now we evaluate diverse behavioral profiles across real-world operational scenarios.

In [ ]:
test_scenarios = [
    {
        "name": "Scenario A: Standard User Workday",
        "features": {
            "failed_logins": 0,
            "encryption_requests": 4,
            "decryption_requests": 3,
            "failed_decryptions": 0,
            "access_frequency": 1.5
        }
    },
    {
        "name": "Scenario B: Occasional Typo / Minor Issue",
        "features": {
            "failed_logins": 1,
            "encryption_requests": 5,
            "decryption_requests": 2,
            "failed_decryptions": 0,
            "access_frequency": 2.2
        }
    },
    {
        "name": "Scenario C: Credential Stuffing / Brute Force",
        "features": {
            "failed_logins": 7,
            "encryption_requests": 1,
            "decryption_requests": 0,
            "failed_decryptions": 0,
            "access_frequency": 9.5
        }
    },
    {
        "name": "Scenario D: Automated Ransomware / Exfiltration Attack",
        "features": {
            "failed_logins": 0,
            "encryption_requests": 2,
            "decryption_requests": 85,
            "failed_decryptions": 12,
            "access_frequency": 45.0
        }
    },
    {
        "name": "Scenario E: Suspicious Decryption Scanner",
        "features": {
            "failed_logins": 2,
            "encryption_requests": 0,
            "decryption_requests": 15,
            "failed_decryptions": 2,
            "access_frequency": 8.5
        }
    }
]

results = []
for s in test_scenarios:
    pred = detector.predict_activity(s["features"])
    results.append({
        "Scenario": s["name"],
        "Failed Logins": s["features"]["failed_logins"],
        "Enc Req": s["features"]["encryption_requests"],
        "Dec Req": s["features"]["decryption_requests"],
        "Failed Dec": s["features"]["failed_decryptions"],
        "Freq (req/min)": s["features"]["access_frequency"],
        "Threat Level": pred["threat_level"],
        "Classification": pred["classification"],
        "Anomaly Score": pred["anomaly_score"],
        "Explanation": pred["explanation"]
    })

df_results = pd.DataFrame(results)
df_results[["Scenario", "Threat Level", "Classification", "Anomaly Score", "Explanation"]]

### 3.1 Anomaly Score Interpretation
- **Higher Anomaly Scores (> 0.0)** indicate standard, predictable behavior conforming to typical user habits.
- **Marginal Anomaly Scores (-0.05 to 0.0)** trigger `SUSPICIOUS` flags with enhanced auditing.
- **Low Negative Anomaly Scores (< -0.10)** or critical threshold violations immediately trigger `POTENTIAL THREAT` isolation.

--- 
## ⚡ Part 4: Cryptographic Performance Benchmark

Let's measure the throughput and latency of symmetric AES-256-GCM vs asymmetric RSA-2048 operations.

In [ ]:
import secrets

sizes_kb = [1, 10, 100, 500, 1024]  # 1 KB to 1 MB
benchmark_data = []

for size in sizes_kb:
    data = secrets.token_bytes(size * 1024)
    
    # Benchmark Hybrid Encryption
    t0 = time.perf_counter()
    pkg = encrypt_hybrid(data)
    t_enc = (time.perf_counter() - t0) * 1000
    
    # Benchmark Hybrid Decryption
    t0 = time.perf_counter()
    res = decrypt_hybrid(pkg["encrypted_file_payload"], pkg["encrypted_aes_key_b64"], pkg["sha256_hash"])
    t_dec = (time.perf_counter() - t0) * 1000
    
    benchmark_data.append({
        "Payload Size (KB)": size,
        "Encryption Time (ms)": round(t_enc, 3),
        "Decryption Time (ms)": round(t_dec, 3),
        "Total Roundtrip (ms)": round(t_enc + t_dec, 3),
        "Integrity Status": "Verified" if res["integrity_verified"] else "Failed"
    })

df_bench = pd.DataFrame(benchmark_data)
df_bench

### Conclusion & System Architecture Summary

The **AI-Driven Hybrid File Encryption System** delivers:
1. **Provable Cryptographic Security:** AES-256-GCM ensures authenticated encryption with zero possibility of undetectable tampering.
2. **Scalable Key Management:** RSA-2048 wraps AES keys securely for multi-user distribution.
3. **Proactive AI Defense:** Scikit-Learn's Isolation Forest continuously inspects request frequencies, failed decryptions, and login anomalies to neutralize threats in real time before data exfiltration occurs.